# Chlorella sp. Multi-Target Machine Learning Optimization
## Bioresource Technology

**Targets:** Biomass (mg/L) | Total Chl (mg/L) | Biomass+Chl (normalized)

**Models:** XGBoost, LightGBM, CatBoost, Ensemble

In [ ]:
# Chlorella sp. Multi-Target Machine Learning Optimization
# Merged & Fixed Script for Bioresource Technology Submission
#
# 修改内容:
#   1. 路径改为读取当前目录CSV (无需硬编码)
#   2. 列名自动对齐 (K2HPO4/KH2PO4 变体名称统一)
#   3. Biomass 单位统一为 mg L-1 (全程保持 mg/L)
#   4. 新增 MOPSO 多目标帕累托优化模块
#   5. 新增实验验证对比表模块
#   6. 合并两个原始脚本所有可视化
#   7. 目标: 干重(mg/L) + 叶绿素(mg/L) + 干重×叶绿素归一化组合


In [ ]:
import sys, os, json, warnings, re
from pathlib import Path
try:
    sys.stdout.reconfigure(encoding='utf-8')
except Exception:
    pass
warnings.filterwarnings('ignore')
os.environ['LOKY_MAX_CPU_COUNT'] = '1'  # 避免 Windows 下 loky 并行警告

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import seaborn as sns
from math import pi
from scipy import stats
from scipy.optimize import differential_evolution

# ML
from sklearn.model_selection import KFold, train_test_split
from sklearn.impute import SimpleImputer
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error
from sklearn.feature_selection import RFECV
from sklearn.pipeline import Pipeline
from sklearn.inspection import PartialDependenceDisplay, partial_dependence
from sklearn.ensemble import VotingRegressor
import joblib
import shap

# 贝叶斯优化
from skopt import gp_minimize
from skopt.space import Real, Integer
from skopt.utils import use_named_args


CSV 放在同目录下即可，无需修改路径

In [ ]:
try:
    BASE_DIR = Path(__file__).resolve().parent
except NameError:
    BASE_DIR = Path.cwd()

DATA_CSV_NAME = '小球藻单因素优化汇总数据1.csv'

def resolve_data_csv(path_or_name=DATA_CSV_NAME):
    """Resolve the uploaded CSV from common run locations."""
    candidates = []
    env_csv = os.environ.get('CHLORELLA_DATA_CSV')
    if env_csv:
        candidates.append(Path(env_csv).expanduser())

    given = Path(path_or_name).expanduser()
    if given.is_absolute():
        candidates.append(given)
    else:
        candidates.extend([
            Path.cwd() / given,
            BASE_DIR / given,
            Path.home() / given,
            Path.home() / 'Desktop' / '论文需要文件' / given,
        ])

    for candidate in candidates:
        if candidate.exists():
            return str(candidate)

    checked = '\n'.join(f'  - {c}' for c in candidates)
    raise FileNotFoundError(f'Cannot find CSV file {path_or_name}. Checked:\n{checked}')

DATA_CSV   = resolve_data_csv(DATA_CSV_NAME)
OUTPUT_DIR = str(BASE_DIR / 'chlorella_results')
os.makedirs(OUTPUT_DIR, exist_ok=True)

OPTIMIZE_DAY = 7     # 固定预测天数（用于优化）
RANDOM_SEED  = 42


In [ ]:
# 图形全局参数


In [ ]:
plt.rcParams['font.family']      = 'Times New Roman'
plt.rcParams['axes.unicode_minus'] = False
plt.rcParams['font.size']        = 11
plt.rcParams['axes.titlesize']   = 13
plt.rcParams['axes.labelsize']   = 11
plt.rcParams['xtick.labelsize']  = 9
plt.rcParams['ytick.labelsize']  = 9
plt.rcParams['legend.fontsize']  = 9
sns.set_style('whitegrid')
sns.set_palette('Set2')


In [ ]:
print('=' * 60)
print('1. 数据加载与预处理')
print('=' * 60)

df = pd.read_csv(DATA_CSV, encoding='utf-8-sig')
print(f'原始数据维度: {df.shape}')
print(f'原始列名: {list(df.columns)}')

# ── 列名标准化映射 ─────────────────────────────────────────
# 解决原CSV中列名含换行符/特殊括号/空格问题
def _norm(s):
    out = str(s).replace('\r', '').replace('\n', '')
    out = out.replace('（', '(').replace('）', ')')
    out = re.sub(r'[·∙・•⋅路]\s*[37]H2O', '', out, flags=re.IGNORECASE)
    out = re.sub(r'\s+', ' ', out).strip()
    while '((' in out:
        out = out.replace('((', '(')
    return out

df.columns = [_norm(c) for c in df.columns]
print(f'数据文件: {DATA_CSV}')
print(f'标准化后列名: {list(df.columns)}')

# ── 特征列与目标列 ─────────────────────────────────────────
# 注意：CSV实际列名标准化后的精确写法
FEATURE_COLS = [
    'Illumination intensity (μmol photons m-2 s-1)',
    'Light time (h)',
    'Dark time (h)',
    'time(d)',
    'temperature(℃)',
    'PH',
    'NH4Cl(g L-1)',
    'NaNO3(g L-1)',
    'CO(NH2)2(g L-1)',
    'MgSO4(g L-1)',
    'CaCl2·2H2O(g L-1)',
    'K2HPO4(mg L-1)',
    'KH2PO4(mg L-1)',
    'trace elements(Standard concentration)',
    'seeding density(OD680)',
    'C4H11NO3(g L-1))',
    'C2H4O2(ml L-1)',
]

TARGET_RAW = {
    'Biomass_raw':       'Biomass (mg L-1)',    # CSV中为mg L-1，保持不变
    'Total_Chl_raw':     'Total Chl (mg L-1)',
}

# 自动匹配列名（模糊匹配以防细微差异）
def find_col(df_cols, keyword):
    """在df列名中找含keyword的第一个列"""
    for c in df_cols:
        if keyword.lower() in c.lower():
            return c
    return None

# 尝试自动对齐特征列
actual_feat_cols = []
for fc in FEATURE_COLS:
    if fc in df.columns:
        actual_feat_cols.append(fc)
    else:
        # 用关键词模糊匹配
        key = fc.split('(')[0].strip()
        matched = find_col(df.columns, key)
        if matched:
            print(f'  列名映射: [{fc}] -> [{matched}]')
            actual_feat_cols.append(matched)
        else:
            raise ValueError(
                f'Missing feature column [{fc}]. Please fix FEATURE_COLS or the CSV header mapping; '
                'do not fill a missing feature column with 0.'
            )

FEATURE_COLS = actual_feat_cols

# Target columns
biomass_col = find_col(df.columns, 'Biomass')

# Total Chl column was renamed from Total Chl a to Total Chl.
# Prefer the corrected exact name, with old spellings kept only as fallback.
chla_col = None
for _cand in [
    'Total Chl (mg L-1)',
    'Total Chl(mg L-1)',
    'Total Chl a (mg L-1)',
    'Total Chl a(mg L-1)',
]:
    if _cand in df.columns:
        chla_col = _cand
        break
if chla_col is None:
    chla_col = find_col(df.columns, 'Total Chl')
if chla_col is None:
    chla_col = find_col(df.columns, 'Chl a')
if chla_col is None:
    raise ValueError('Cannot find target column Total Chl (mg L-1); please check the CSV header')

print(f'\nTarget columns: Biomass={biomass_col}, Total Chl={chla_col}')

# 提取特征和目标
X_raw_df = df[FEATURE_COLS].copy()
X_raw    = X_raw_df.values.astype(float)

# Biomass: 保持 mg L-1（不做单位转换）
target_biomass = df[biomass_col].values.astype(float)
print('Biomass 单位保持 mg L-1')

# Total chlorophyll (mg L-1, unchanged)
target_chl = df[chla_col].values.astype(float)

# 缺失值处理
def fill_nan_col(arr, name):
    if np.isnan(arr).all():
        raise ValueError(f'Target column {name} is entirely missing; cannot train model')
    if np.isnan(arr).any():
        print(f'  Target column {name} has missing values; filling with target mean')
        arr = np.where(np.isnan(arr), np.nanmean(arr), arr)
    return arr

target_biomass = fill_nan_col(target_biomass, biomass_col)
target_chl     = fill_nan_col(target_chl, chla_col)

# Feature NaNs are imputed inside each model pipeline using train-only medians.
X_df = pd.DataFrame(X_raw, columns=FEATURE_COLS)

# Validate the 24-hour photoperiod constraint in the input data.
LIGHT_COL = 'Light time (h)'
DARK_COL = 'Dark time (h)'
if LIGHT_COL in X_df.columns and DARK_COL in X_df.columns:
    photoperiod_sum = X_df[LIGHT_COL] + X_df[DARK_COL]
    bad_mask = ~np.isclose(photoperiod_sum, 24.0, atol=1e-6)
    if bad_mask.any():
        print(f'Warning: {bad_mask.sum()} rows do not satisfy Light time + Dark time = 24 h')
    else:
        print('Photoperiod check passed: Light time + Dark time = 24 h')
else:
    raise ValueError('Light time and Dark time columns are required for the 24-hour photoperiod constraint')

# Validate mutually exclusive nitrogen sources: only one of the three should be present in each row.
NITROGEN_COLS = ['NH4Cl(g L-1)', 'NaNO3(g L-1)', 'CO(NH2)2(g L-1)']
missing_n_cols = [c for c in NITROGEN_COLS if c not in X_df.columns]
if missing_n_cols:
    raise ValueError(f'Missing nitrogen source columns: {missing_n_cols}')
n_positive = (X_df[NITROGEN_COLS].fillna(0) > 1e-12).sum(axis=1)
if (n_positive > 1).any():
    print(f'Warning: {(n_positive > 1).sum()} rows contain more than one nitrogen source')
else:
    print('Nitrogen source check passed: at most one nitrogen source is present per row')

# Validate phosphorus source columns. Cultivation optimization enforces
# K2HPO4/KH2PO4 as a mutually exclusive source choice.
PHOSPHORUS_COLS = ['K2HPO4(mg L-1)', 'KH2PO4(mg L-1)']
missing_p_cols = [c for c in PHOSPHORUS_COLS if c not in X_df.columns]
if missing_p_cols:
    raise ValueError(f'Missing phosphorus source columns: {missing_p_cols}')
p_positive = (X_df[PHOSPHORUS_COLS].fillna(0) > 1e-12).sum(axis=1)
if (p_positive > 1).any():
    print(f'Data note: {(p_positive > 1).sum()} rows contain both phosphorus columns; optimization will still enforce one phosphorus source')
else:
    print('Phosphorus source check passed: at most one phosphorus source is present per row')

# Use one fixed split for all targets, plots, model selection, and final testing.
ALL_IDX = np.arange(len(X_df))
TRAIN_VAL_IDX, TEST_IDX = train_test_split(
    ALL_IDX, test_size=0.2, random_state=RANDOM_SEED
)
TRAIN_IDX, VAL_IDX = train_test_split(
    TRAIN_VAL_IDX, test_size=0.2, random_state=RANDOM_SEED
)
print(f'Data split: train={len(TRAIN_IDX)}, val={len(VAL_IDX)}, test={len(TEST_IDX)}')

# Features shown in influence-factor figures. Keep time(d) in the model,
# but exclude it from later interpretation plots because optimization fixes the day.
DAY_COL = 'time(d)'
INFLUENCE_FEATURE_COLS = [f for f in FEATURE_COLS if f != DAY_COL]

# Composite target normalization uses train-set min/max only.
def norm01_target_train(arr, train_idx):
    lo, hi = arr[train_idx].min(), arr[train_idx].max()
    if hi - lo < 1e-12:
        return np.zeros_like(arr)
    return (arr - lo) / (hi - lo)

biomass_norm = norm01_target_train(target_biomass, TRAIN_IDX)
chl_norm     = norm01_target_train(target_chl, TRAIN_IDX)
target_combined = biomass_norm + chl_norm

TARGETS = {
    'Biomass (mg L⁻¹)':      target_biomass,
    'Total Chl (mg L⁻¹)':    target_chl,
    'Biomass+Chl (normalized)': target_combined,
}

print(f'\n最终特征矩阵: {X_df.shape}')
print('\n各目标统计:')
for name, arr in TARGETS.items():
    print(f'  {name}: min={arr.min():.4f}, max={arr.max():.4f}, mean={arr.mean():.4f}')


In [ ]:
print('\n' + '=' * 60)
print('2. 模型配置')
print('=' * 60)

available_models = {}
try:
    from xgboost import XGBRegressor
    available_models['XGBoost'] = XGBRegressor
    print('XGBoost 已加载')
except ImportError:
    print('XGBoost 未安装，跳过')

try:
    import lightgbm as lgb
    available_models['LightGBM'] = lgb.LGBMRegressor
    print('LightGBM 已加载')
except ImportError:
    print('LightGBM 未安装，跳过')

try:
    import catboost as cb
    available_models['CatBoost'] = cb.CatBoostRegressor
    print('CatBoost 已加载')
except ImportError:
    print('CatBoost 未安装，跳过')

if not available_models:
    raise ImportError('至少需要安装 xgboost、lightgbm 或 catboost 之一')

MODEL_CONFIG = {}
if 'XGBoost' in available_models:
    MODEL_CONFIG['XGBoost'] = {
        'class': XGBRegressor,
        'space': [
            Integer(3, 20,   name='max_depth'),
            Integer(100, 500, name='n_estimators'),
            Real(0.01, 0.3,  prior='log-uniform', name='learning_rate'),
            Real(1e-8, 0.1,  prior='log-uniform', name='reg_alpha'),
            Real(1e-8, 0.1,  prior='log-uniform', name='reg_lambda'),
        ],
        'fixed': {'random_state': RANDOM_SEED, 'objective': 'reg:squarederror', 'n_jobs': -1},
    }
if 'LightGBM' in available_models:
    MODEL_CONFIG['LightGBM'] = {
        'class': lgb.LGBMRegressor,
        'space': [
            Integer(3, 20,  name='num_leaves'),
            Integer(100, 500, name='n_estimators'),
            Real(0.01, 0.3, prior='log-uniform', name='learning_rate'),
            Real(0.5, 1.0,  name='feature_fraction'),
            Real(0.5, 1.0,  name='bagging_fraction'),
            Integer(1, 10,  name='bagging_freq'),
        ],
        'fixed': {'random_state': RANDOM_SEED, 'objective': 'regression',
                  'metric': 'rmse', 'verbosity': -1},
    }
if 'CatBoost' in available_models:
    MODEL_CONFIG['CatBoost'] = {
        'class': cb.CatBoostRegressor,
        'space': [
            Integer(3, 10,  name='depth'),
            Integer(100, 500, name='iterations'),
            Real(0.01, 0.3, prior='log-uniform', name='learning_rate'),
            Real(1e-8, 0.1, prior='log-uniform', name='l2_leaf_reg'),
        ],
        'fixed': {'random_seed': RANDOM_SEED, 'loss_function': 'RMSE',
                  'verbose': False, 'early_stopping_rounds': 50},
    }

print(f'将对比模型: {list(MODEL_CONFIG.keys())}')


In [ ]:
def train_one_model(model_name, config, X_tr, y_tr, X_va, y_va,
                    X_te, y_te, feat_names, out_dir, tgt_name):
    """RFECV特征选择 + 贝叶斯超参优化 + 评估 + SHAP"""
    try:
        from xgboost.callback import EarlyStopping as _XGBES
    except ImportError:
        class _XGBES:
            def __init__(self, **kw): pass

    print(f'\n--- 训练 {model_name} | 目标: {tgt_name} ---')
    cls   = config['class']
    space = config['space']
    fixed = config['fixed']

    # 1) Train-only imputation + RFECV feature selection
    imputer = SimpleImputer(strategy='median')
    X_tr_i = imputer.fit_transform(X_tr)
    X_va_i = imputer.transform(X_va)
    X_te_i = imputer.transform(X_te)

    base     = cls(**{k: v for k, v in fixed.items()
                      if k not in ('early_stopping_rounds',)})
    rfecv_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    selector = RFECV(estimator=base, step=1, scoring='r2', cv=rfecv_cv, n_jobs=1)
    selector.fit(X_tr_i, y_tr)
    sel_feats = [feat_names[i] for i in selector.get_support(indices=True)]
    print(f'  RFECV selected features: {len(sel_feats)}/{len(feat_names)}')
    X_tr_s = selector.transform(X_tr_i)
    X_va_s = selector.transform(X_va_i)
    X_te_s = selector.transform(X_te_i)

    # 2) 贝叶斯优化
    @use_named_args(space)
    def objective(**params):
        scores = []
        kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
        for ti, vi in kf.split(X_tr_s):
            m = cls(**fixed, **params)
            try:
                if model_name == 'XGBoost':
                    m.fit(X_tr_s[ti], y_tr[ti],
                          eval_set=[(X_tr_s[vi], y_tr[vi])],
                          callbacks=[_XGBES(rounds=30, metric_name='rmse')],
                          verbose=False)
                elif model_name == 'LightGBM':
                    m.fit(X_tr_s[ti], y_tr[ti],
                          eval_set=[(X_tr_s[vi], y_tr[vi])],
                          callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
                elif model_name == 'CatBoost':
                    m.fit(X_tr_s[ti], y_tr[ti],
                          eval_set=[(X_tr_s[vi], y_tr[vi])], verbose=False)
                else:
                    m.fit(X_tr_s[ti], y_tr[ti])
            except TypeError:
                m.fit(X_tr_s[ti], y_tr[ti])
            scores.append(r2_score(y_tr[vi], m.predict(X_tr_s[vi])))
        return -np.mean(scores)

    res    = gp_minimize(objective, space, n_calls=15,
                         random_state=RANDOM_SEED, verbose=False)
    best_p = {d.name: (v.item() if hasattr(v, 'item') else v)
              for d, v in zip(space, res.x)}
    print(f'  最佳超参: {best_p}')

    # 3) 用最佳参数训练最终模型
    final = cls(**fixed, **best_p)
    try:
        if model_name == 'XGBoost':
            final.fit(X_tr_s, y_tr, eval_set=[(X_va_s, y_va)],
                      callbacks=[_XGBES(rounds=30, metric_name='rmse')], verbose=False)
        elif model_name == 'LightGBM':
            final.fit(X_tr_s, y_tr, eval_set=[(X_va_s, y_va)],
                      callbacks=[lgb.early_stopping(30), lgb.log_evaluation(0)])
        elif model_name == 'CatBoost':
            final.fit(X_tr_s, y_tr, eval_set=[(X_va_s, y_va)], verbose=False)
        else:
            final.fit(X_tr_s, y_tr)
    except TypeError:
        final.fit(X_tr_s, y_tr)

    # 4) 评估
    pred_tr = final.predict(X_tr_s)
    pred_va = final.predict(X_va_s)
    pred_te = final.predict(X_te_s)
    metrics = {
        'train_r2':   r2_score(y_tr, pred_tr),
        'val_r2':     r2_score(y_va, pred_va),
        'test_r2':    r2_score(y_te, pred_te),
        'train_rmse': np.sqrt(mean_squared_error(y_tr, pred_tr)),
        'val_rmse':   np.sqrt(mean_squared_error(y_va, pred_va)),
        'test_rmse':  np.sqrt(mean_squared_error(y_te, pred_te)),
        'train_mae':  mean_absolute_error(y_tr, pred_tr),
        'val_mae':    mean_absolute_error(y_va, pred_va),
        'test_mae':   mean_absolute_error(y_te, pred_te),
    }
    print(f'  R2 train={metrics["train_r2"]:.3f}  val={metrics["val_r2"]:.3f}  '
          f'test={metrics["test_r2"]:.3f}  RMSE_te={metrics["test_rmse"]:.4f}')

    # 5) 保存 pipeline
    pipeline  = Pipeline([('imputer', imputer), ('selector', selector), ('model', final)])
    pkl_path  = os.path.join(out_dir, f'{tgt_name}_{model_name}_pipeline.pkl')
    joblib.dump(pipeline, pkl_path)

    # 6) SHAP
    shap_df = None
    try:
        X_tr_s_f  = np.array(X_tr_s[:200], dtype=np.float64)
        explainer = shap.TreeExplainer(final)
        shap_vals = explainer(X_tr_s_f)
        mean_shap = np.abs(shap_vals.values).mean(axis=0)
        shap_df   = pd.DataFrame({'Feature': sel_feats, 'Mean|SHAP|': mean_shap})
        shap_df   = shap_df.sort_values('Mean|SHAP|', ascending=False)
    except Exception as e:
        print(f'  SHAP 分析失败: {e}')

    return {'model_name': model_name, **metrics,
            'selected_features': sel_feats,
            'best_params': best_p,
            'pipeline_path': pkl_path,
            'shap_df': shap_df}


In [ ]:
def train_ensemble_model(X_tr, y_tr, X_va, y_va, X_te, y_te,
                         feat_names, out_dir, tgt_name):
    """RFECV特征选择 + 分别优化XGBoost/CatBoost + VotingRegressor集成"""
    print(f'\n--- 训练 Ensemble (XGBoost+CatBoost) | 目标: {tgt_name} ---')

    from xgboost import XGBRegressor
    import catboost as cb
    from xgboost.callback import EarlyStopping as _XGBES

    # 1) Train-only imputation + RFECV feature selection (XGBoost selector)
    imputer = SimpleImputer(strategy='median')
    X_tr_i = imputer.fit_transform(X_tr)
    X_va_i = imputer.transform(X_va)
    X_te_i = imputer.transform(X_te)

    base_selector = XGBRegressor(
        random_state=RANDOM_SEED, objective='reg:squarederror',
        n_estimators=100, max_depth=6, learning_rate=0.1)
    rfecv_cv = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
    selector = RFECV(estimator=base_selector, step=1, scoring='r2', cv=rfecv_cv, n_jobs=1)
    selector.fit(X_tr_i, y_tr)
    sel_feats = [feat_names[i] for i in selector.get_support(indices=True)]
    print(f'  RFECV selected features: {len(sel_feats)}/{len(feat_names)}')
    X_tr_s = selector.transform(X_tr_i)
    X_va_s = selector.transform(X_va_i)
    X_te_s = selector.transform(X_te_i)

    # 2) 分别贝叶斯优化 XGBoost 和 CatBoost
    xgb_space = MODEL_CONFIG['XGBoost']['space']
    xgb_fixed = MODEL_CONFIG['XGBoost']['fixed']

    @use_named_args(xgb_space)
    def xgb_objective(**params):
        scores = []
        kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
        for ti, vi in kf.split(X_tr_s):
            m = XGBRegressor(**xgb_fixed, **params)
            try:
                m.fit(X_tr_s[ti], y_tr[ti],
                      eval_set=[(X_tr_s[vi], y_tr[vi])],
                      callbacks=[_XGBES(rounds=30, metric_name='rmse')],
                      verbose=False)
            except TypeError:
                m.fit(X_tr_s[ti], y_tr[ti])
            scores.append(r2_score(y_tr[vi], m.predict(X_tr_s[vi])))
        return -np.mean(scores)

    print('  优化 XGBoost 超参...')
    xgb_res = gp_minimize(xgb_objective, xgb_space, n_calls=15,
                          random_state=RANDOM_SEED, verbose=False)
    xgb_best = {d.name: (v.item() if hasattr(v, 'item') else v)
                for d, v in zip(xgb_space, xgb_res.x)}

    cb_space = MODEL_CONFIG['CatBoost']['space']
    cb_fixed = MODEL_CONFIG['CatBoost']['fixed']

    @use_named_args(cb_space)
    def cb_objective(**params):
        scores = []
        kf = KFold(n_splits=5, shuffle=True, random_state=RANDOM_SEED)
        for ti, vi in kf.split(X_tr_s):
            m = cb.CatBoostRegressor(**cb_fixed, **params)
            try:
                m.fit(X_tr_s[ti], y_tr[ti],
                      eval_set=[(X_tr_s[vi], y_tr[vi])], verbose=False)
            except TypeError:
                m.fit(X_tr_s[ti], y_tr[ti])
            scores.append(r2_score(y_tr[vi], m.predict(X_tr_s[vi])))
        return -np.mean(scores)

    print('  优化 CatBoost 超参...')
    cb_res = gp_minimize(cb_objective, cb_space, n_calls=15,
                         random_state=RANDOM_SEED, verbose=False)
    cb_best = {d.name: (v.item() if hasattr(v, 'item') else v)
               for d, v in zip(cb_space, cb_res.x)}

    print(f'  XGBoost 最佳超参: {xgb_best}')
    print(f'  CatBoost 最佳超参: {cb_best}')

    # 3) 创建 VotingRegressor
    xgb_final = XGBRegressor(**xgb_fixed, **xgb_best)
    cb_final  = cb.CatBoostRegressor(**cb_fixed, **cb_best)
    ensemble  = VotingRegressor(estimators=[('xgb', xgb_final), ('cb', cb_final)])

    try:
        ensemble.fit(X_tr_s, y_tr)
    except Exception as e:
        print(f'  VotingRegressor 训练失败: {e}')
        return None

    # 4) 评估
    pred_tr = ensemble.predict(X_tr_s)
    pred_va = ensemble.predict(X_va_s)
    pred_te = ensemble.predict(X_te_s)
    metrics = {
        'train_r2':   r2_score(y_tr, pred_tr),
        'val_r2':     r2_score(y_va, pred_va),
        'test_r2':    r2_score(y_te, pred_te),
        'train_rmse': np.sqrt(mean_squared_error(y_tr, pred_tr)),
        'val_rmse':   np.sqrt(mean_squared_error(y_va, pred_va)),
        'test_rmse':  np.sqrt(mean_squared_error(y_te, pred_te)),
        'train_mae':  mean_absolute_error(y_tr, pred_tr),
        'val_mae':    mean_absolute_error(y_va, pred_va),
        'test_mae':   mean_absolute_error(y_te, pred_te),
    }
    print(f'  R2 train={metrics["train_r2"]:.3f}  val={metrics["val_r2"]:.3f}  '
          f'test={metrics["test_r2"]:.3f}  RMSE_te={metrics["test_rmse"]:.4f}')

    # 5) 保存 pipeline
    pipeline = Pipeline([('imputer', imputer), ('selector', selector), ('model', ensemble)])
    pkl_path = os.path.join(out_dir, f'{tgt_name}_Ensemble_pipeline.pkl')
    joblib.dump(pipeline, pkl_path)

    # 6) SHAP (用XGBoost子模型近似)
    shap_df = None
    try:
        X_tr_s_f  = np.array(X_tr_s[:200], dtype=np.float64)
        explainer = shap.TreeExplainer(xgb_final.fit(X_tr_s, y_tr))
        shap_vals = explainer(X_tr_s_f)
        mean_shap = np.abs(shap_vals.values).mean(axis=0)
        shap_df   = pd.DataFrame({'Feature': sel_feats, 'Mean|SHAP|': mean_shap})
        shap_df   = shap_df.sort_values('Mean|SHAP|', ascending=False)
    except Exception as e:
        print(f'  SHAP 分析失败: {e}')

    return {'model_name': 'Ensemble', **metrics,
            'selected_features': sel_feats,
            'best_params': {'XGBoost': xgb_best, 'CatBoost': cb_best},
            'pipeline_path': pkl_path,
            'shap_df': shap_df}


In [ ]:
print('\n' + '=' * 60)
print('3. 多目标 × 多模型训练')
print('=' * 60)

all_results    = {}
best_per_target = {}

for tgt_name, y_arr in TARGETS.items():
    print(f'\n{"="*50}\n目标: {tgt_name}\n{"="*50}')
    X      = X_df.values
    X_tr, X_va, X_te = X[TRAIN_IDX], X[VAL_IDX], X[TEST_IDX]
    y_tr, y_va, y_te = y_arr[TRAIN_IDX], y_arr[VAL_IDX], y_arr[TEST_IDX]
    tgt_results = {}
    for mname, mconfig in MODEL_CONFIG.items():
        try:
            res = train_one_model(mname, mconfig, X_tr, y_tr, X_va, y_va,
                                  X_te, y_te, FEATURE_COLS, OUTPUT_DIR, tgt_name)
            tgt_results[mname] = res
        except Exception as e:
            print(f'  {mname} 训练失败: {e}')

    # Ensemble 模型 (需要 XGBoost 和 CatBoost 已训练)
    if 'XGBoost' in available_models and 'CatBoost' in available_models:
        try:
            ens_res = train_ensemble_model(X_tr, y_tr, X_va, y_va,
                                           X_te, y_te, FEATURE_COLS,
                                           OUTPUT_DIR, tgt_name)
            if ens_res:
                tgt_results['Ensemble'] = ens_res
        except Exception as e:
            print(f'  Ensemble 训练失败: {e}')

    all_results[tgt_name] = tgt_results

    if tgt_results:
        best_name = max(tgt_results, key=lambda k: tgt_results[k]['val_r2'])
        best_per_target[tgt_name] = tgt_results[best_name]
        best_per_target[tgt_name]['target'] = tgt_name
        print(f'\n  => Best model: {best_name} '
              f'(Val R2={tgt_results[best_name]["val_r2"]:.4f}, '
              f'Test R2={tgt_results[best_name]["test_r2"]:.4f})')

# Save model summary CSV
summary_rows = []
for tgt, res_dict in all_results.items():
    for mn, r in res_dict.items():
        summary_rows.append({
            'Target': tgt, 'Model': mn,
            'Train_R2':   r['train_r2'],  'Val_R2':   r['val_r2'],  'Test_R2':   r['test_r2'],
            'Train_RMSE': r['train_rmse'],'Val_RMSE': r['val_rmse'],'Test_RMSE': r['test_rmse'],
            'Train_MAE':  r['train_mae'], 'Val_MAE':  r['val_mae'], 'Test_MAE':  r['test_mae'],
        })
summary_df = pd.DataFrame(summary_rows)
summary_df.to_csv(os.path.join(OUTPUT_DIR, 'model_comparison_all.csv'),
                  index=False, encoding='utf-8-sig')
print('\n模型汇总:')
print(summary_df.to_string(index=False))


In [ ]:
print('\n' + '=' * 60)
print('4. PCC 相关系数矩阵')
print('=' * 60)

all_data = X_df[INFLUENCE_FEATURE_COLS].copy()
all_data['Biomass']       = target_biomass
all_data['Total Chl']     = target_chl
all_data['Biomass+Chl']   = target_combined

short_names = {}
for c in INFLUENCE_FEATURE_COLS:
    s = c.split('(')[0].strip()
    short_names[c] = s[:14] if len(s) > 14 else s
short_names['Biomass']       = 'Biomass'
short_names['Total Chl']     = 'Total Chl'
short_names['Biomass+Chl']   = 'Biomass+Chl'

corr = all_data.rename(columns=short_names).corr(method='pearson')
fig, ax = plt.subplots(figsize=(18, 14))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask,
            cmap='RdBu_r', vmin=-1, vmax=1, center=0, square=True,
            linewidths=0.6, linecolor='white',
            annot=True, fmt='.2f',
            annot_kws={'size': 7, 'fontweight': 'bold'},
            cbar_kws={'shrink': 0.75, 'label': 'Pearson r',
                      'orientation': 'vertical', 'pad': 0.02}, ax=ax)
ax.set_title('Pearson Correlation Coefficient Matrix',
             fontsize=16, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'Fig_PCC_heatmap.png'), dpi=350, bbox_inches='tight')
plt.close()
print('PCC 热力图已保存')


In [ ]:
print('\n' + '=' * 60)
print('5. 数据分布直方图')
print('=' * 60)

ncols_d = 5
nrows_d = int(np.ceil((len(INFLUENCE_FEATURE_COLS) + 3) / ncols_d))
fig, axes = plt.subplots(nrows_d, ncols_d, figsize=(22, nrows_d * 3.8))
axes = axes.flatten()
pal = sns.color_palette('mako', len(INFLUENCE_FEATURE_COLS))
for i, col in enumerate(INFLUENCE_FEATURE_COLS):
    ax  = axes[i]
    vals = X_df[col].values
    vals = vals[~np.isnan(vals)]
    ax.hist(vals, bins=30, color=pal[i], edgecolor='white', alpha=0.8)
    try:
        from scipy.stats import gaussian_kde
        if len(vals) > 2 and vals.std() > 0:
            kde_x = np.linspace(vals.min(), vals.max(), 200)
            kde = gaussian_kde(vals)
            ax2 = ax.twinx()
            ax2.plot(kde_x, kde(kde_x), color='#E74C3C', lw=1.5, alpha=0.7)
            ax2.set_ylabel('')
            ax2.set_yticks([])
            ax2.spines['right'].set_visible(False)
    except Exception:
        pass
    ax.set_title(col.split('(')[0].strip()[:18], fontsize=9, fontweight='bold')
    ax.set_ylabel('Count', fontsize=8)
    ax.tick_params(labelsize=7)
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)

tgt_colors = ['#E74C3C', '#3498DB', '#2ECC71']
for j, (name, arr) in enumerate(TARGETS.items()):
    idx = len(INFLUENCE_FEATURE_COLS) + j
    if idx < len(axes):
        ax = axes[idx]
        ax.hist(arr, bins=30, color=tgt_colors[j], edgecolor='white', alpha=0.8)
        ax.set_title(name, fontsize=9, fontweight='bold')
        ax.set_ylabel('Count', fontsize=8)
        ax.tick_params(labelsize=7)
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)

for k in range(len(INFLUENCE_FEATURE_COLS) + 3, len(axes)):
    axes[k].axis('off')

plt.suptitle('Distribution of Input Features and Target Variables',
             fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'Fig_data_distribution.png'),
            dpi=350, bbox_inches='tight')
plt.close()
print('数据分布图已保存')


In [ ]:
print('\n' + '=' * 60)
print('6. 预测性能散点图 + 残差图')
print('=' * 60)

for tgt_name, y_arr in TARGETS.items():
    if tgt_name not in best_per_target:
        continue
    best  = best_per_target[tgt_name]
    bname = best['model_name']
    pipe  = joblib.load(best['pipeline_path'])

    X      = X_df.values
    X_tr, X_te = X[TRAIN_IDX], X[TEST_IDX]
    y_tr, y_te = y_arr[TRAIN_IDX], y_arr[TEST_IDX]
    pred_tr = pipe.predict(X_tr)
    pred_te = pipe.predict(X_te)
    r2_tr   = r2_score(y_tr, pred_tr);  r2_te   = r2_score(y_te, pred_te)
    rmse_tr = np.sqrt(mean_squared_error(y_tr, pred_tr))
    rmse_te = np.sqrt(mean_squared_error(y_te, pred_te))
    mae_te  = mean_absolute_error(y_te, pred_te)

    # --- 散点图 ---
    fig, axes = plt.subplots(1, 2, figsize=(13, 5.5))
    for ax, obs, pred, r2, rmse, label, color in [
        (axes[0], y_tr, pred_tr, r2_tr, rmse_tr, f'Training (n={len(y_tr)})', '#3498DB'),
        (axes[1], y_te, pred_te, r2_te, rmse_te, f'Testing (n={len(y_te)})',  '#E74C3C'),
    ]:
        ax.scatter(obs, pred, alpha=0.5, edgecolors='white',
                   linewidth=0.3, c=color, s=35, zorder=3)
        lims = [min(obs.min(), pred.min()) * 0.92,
                max(obs.max(), pred.max()) * 1.08]
        ax.plot(lims, lims, 'k--', lw=2, label='y = x', zorder=4)
        # 回归线
        z = np.polyfit(obs, pred, 1)
        p = np.poly1d(z)
        x_line = np.linspace(lims[0], lims[1], 100)
        ax.plot(x_line, p(x_line), '-', color=color, lw=1.5, alpha=0.6,
                label=f'Fit: y={z[0]:.3f}x+{z[1]:.3f}', zorder=4)
        ax.set_xlim(lims); ax.set_ylim(lims)
        ax.set_xlabel('Observed', fontweight='bold', fontsize=11)
        ax.set_ylabel('Predicted', fontweight='bold', fontsize=11)
        ax.set_title(label, fontweight='bold', fontsize=12)
        ax.text(0.05, 0.93, f'R² = {r2:.3f}\nRMSE = {rmse:.4f}',
                transform=ax.transAxes, fontsize=10, va='top',
                bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                          edgecolor='#BDBDBD', alpha=0.9))
        ax.legend(fontsize=8, loc='lower right')
        ax.set_aspect('equal')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3, linestyle='--')
    plt.suptitle(f'{tgt_name} — {bname}', fontsize=14, fontweight='bold')
    plt.tight_layout()
    tag = tgt_name.replace('⁻', '-').replace(' ', '_').replace('/', '_')
    plt.savefig(os.path.join(OUTPUT_DIR, f'Fig_scatter_{tag}.png'),
                dpi=350, bbox_inches='tight')
    plt.close()

    # --- 残差图 ---
    residuals = y_te - pred_te
    fig, axes = plt.subplots(1, 2, figsize=(13, 5))
    # Residuals vs Predicted
    axes[0].scatter(pred_te, residuals, alpha=0.55, edgecolors='white',
                    linewidth=0.3, c='#9B59B6', s=35, zorder=3)
    axes[0].axhline(0, color='#E74C3C', linestyle='--', lw=2)
    axes[0].fill_between([pred_te.min(), pred_te.max()],
                         -rmse_te, rmse_te, alpha=0.08, color='#E74C3C',
                         label=f'±1 RMSE ({rmse_te:.4f})')
    axes[0].set_xlabel('Predicted', fontweight='bold', fontsize=11)
    axes[0].set_ylabel('Residual', fontweight='bold', fontsize=11)
    axes[0].set_title('Residuals vs Predicted', fontweight='bold', fontsize=12)
    axes[0].legend(fontsize=9)
    axes[0].spines['top'].set_visible(False)
    axes[0].spines['right'].set_visible(False)
    axes[0].grid(True, alpha=0.3, linestyle='--')
    # Residual distribution with normal fit
    axes[1].hist(residuals, bins=25, color='#9B59B6', edgecolor='white',
                 alpha=0.75, density=True)
    from scipy.stats import norm
    mu, sigma = norm.fit(residuals)
    x_norm = np.linspace(residuals.min(), residuals.max(), 100)
    axes[1].plot(x_norm, norm.pdf(x_norm, mu, sigma), 'r-', lw=2,
                 label=f'Normal fit\nμ={mu:.4f}, σ={sigma:.4f}')
    axes[1].axvline(0, color='black', linestyle='--', lw=1.5, alpha=0.7)
    axes[1].set_xlabel('Residual', fontweight='bold', fontsize=11)
    axes[1].set_ylabel('Density', fontweight='bold', fontsize=11)
    axes[1].set_title('Residual Distribution', fontweight='bold', fontsize=12)
    axes[1].legend(fontsize=9)
    axes[1].spines['top'].set_visible(False)
    axes[1].spines['right'].set_visible(False)
    axes[1].grid(True, alpha=0.3, linestyle='--')
    plt.suptitle(f'{tgt_name} — Residual Analysis', fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'Fig_residual_{tag}.png'),
                dpi=350, bbox_inches='tight')
    plt.close()
    print(f'  {tgt_name} ({bname}): R²_test={r2_te:.4f}, RMSE={rmse_te:.4f}, MAE={mae_te:.4f}')


In [ ]:
print('\n' + '=' * 60)
print('7. SHAP 特征重要性')
print('=' * 60)

shap_results = {}  # 保存各目标的SHAP均值，供后续帕累托分析使用

for tgt_name, y_arr in TARGETS.items():
    if tgt_name not in best_per_target:
        continue
    best  = best_per_target[tgt_name]
    bname = best['model_name']
    pipe  = joblib.load(best['pipeline_path'])
    imp   = pipe.named_steps.get('imputer')
    sel   = pipe.named_steps['selector']
    model = pipe.named_steps['model']
    sel_feats = [FEATURE_COLS[i] for i in sel.get_support(indices=True)]
    keep_idx = [i for i, f in enumerate(sel_feats) if f != DAY_COL]
    plot_feats = [sel_feats[i] for i in keep_idx]
    tag   = tgt_name.replace('⁻', '-').replace(' ', '_').replace('/', '_')

    X_tr = X_df.values[TRAIN_IDX]
    X_tr_i = imp.transform(X_tr) if imp is not None else X_tr
    X_tr_s = np.array(sel.transform(X_tr_i)[:200], dtype=np.float64)

    try:
        # 对于Ensemble模型，使用XGBoost子模型进行SHAP分析
        shap_model = model
        if bname == 'Ensemble' and hasattr(model, 'estimators_'):
            shap_model = model.estimators_[0]  # XGBoost sub-model
            print(f'  Ensemble SHAP: 使用 XGBoost 子模型')

        explainer = shap.TreeExplainer(shap_model)
        sv        = explainer(X_tr_s)

        # 蜂群图
        fig, _ = plt.subplots(figsize=(10, max(4, len(plot_feats) * 0.42)))
        shap.summary_plot(sv.values[:, keep_idx], X_tr_s[:, keep_idx], feature_names=plot_feats,
                          show=False, max_display=17, plot_size=None)
        plt.title(f'{tgt_name} — SHAP Bee Swarm ({bname})',
                  fontsize=13, fontweight='bold', pad=10)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f'Fig_SHAP_bee_{tag}.png'),
                    dpi=350, bbox_inches='tight')
        plt.close()

        # 条形图
        mean_shap = np.abs(sv.values[:, keep_idx]).mean(axis=0)
        sdf = pd.DataFrame({'Feature': plot_feats, 'Mean|SHAP|': mean_shap})
        sdf = sdf.sort_values('Mean|SHAP|', ascending=True)
        shap_results[tgt_name] = sdf  # 保存

        fig, ax = plt.subplots(figsize=(10, max(4, len(sdf) * 0.42)))
        colors_s = sns.color_palette('flare', len(sdf))
        bars = ax.barh(range(len(sdf)), sdf['Mean|SHAP|'],
                       color=colors_s, edgecolor='white', height=0.7)
        ax.set_yticks(range(len(sdf)))
        ax.set_yticklabels(sdf['Feature'], fontsize=9)
        ax.set_xlabel('Mean |SHAP value|', fontweight='bold', fontsize=11)
        ax.set_title(f'{tgt_name} — Feature Importance ({bname})',
                     fontsize=13, fontweight='bold')
        for ii, v in enumerate(sdf['Mean|SHAP|']):
            ax.text(v + max(sdf['Mean|SHAP|']) * 0.01, ii,
                    f'{v:.4f}', va='center', fontsize=8, fontweight='bold')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(axis='x', alpha=0.3, linestyle='--')
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, f'Fig_SHAP_bar_{tag}.png'),
                    dpi=350, bbox_inches='tight')
        plt.close()
        print(f'  {tgt_name} SHAP 完成')

    except Exception as e:
        print(f'  {tgt_name} SHAP 失败: {e}')


In [ ]:
print('\n' + '=' * 60)
print('8. 1D PDP + ICE 图')
print('=' * 60)

for tgt_name, y_arr in TARGETS.items():
    if tgt_name not in best_per_target:
        continue
    best  = best_per_target[tgt_name]
    pipe  = joblib.load(best['pipeline_path'])
    sel   = pipe.named_steps['selector']
    sel_indices = sel.get_support(indices=True)
    sel_feats   = [FEATURE_COLS[i] for i in sel_indices if FEATURE_COLS[i] != DAY_COL]
    tag   = tgt_name.replace('⁻', '-').replace(' ', '_').replace('/', '_')

    # 按SHAP重要性排序
    if tgt_name in shap_results:
        order = shap_results[tgt_name].sort_values('Mean|SHAP|', ascending=False)['Feature'].tolist()
        avail = [f for f in order if f in sel_feats]
    else:
        avail = sel_feats

    if not avail:
        continue

    n     = len(avail)
    ncols = min(4, n)
    nrows = int(np.ceil(n / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(5 * ncols, 4.2 * nrows))
    if n == 1:
        axes = np.array([[axes]])
    axes = axes.flatten() if n > 1 else axes.flatten()

    for i, feat in enumerate(avail):
        idx_global = FEATURE_COLS.index(feat)
        ax = axes[i]
        try:
            PartialDependenceDisplay.from_estimator(
                pipe, X_df, [idx_global], kind='both', ax=ax,
                grid_resolution=50, subsample=80,
                ice_lines_kw={'alpha': 0.08, 'color': '#85C1E9'},
                pd_line_kw={'color': '#E74C3C', 'linewidth': 2.5}
            )
            ax.set_title(feat.split('(')[0].strip()[:22],
                         fontsize=10, fontweight='bold')
            ax.set_xlabel('')
            ax.set_ylabel('Partial Dependence', fontsize=8)
            ax.grid(True, alpha=0.2, linestyle='--')
        except Exception as e:
            ax.text(0.5, 0.5, 'Error', ha='center', va='center',
                    transform=ax.transAxes, fontsize=8)
            ax.set_title(feat[:22], fontsize=9)

    for j in range(n, len(axes)):
        axes[j].axis('off')

    plt.suptitle(f'{tgt_name} — 1D PDP + ICE (by SHAP rank)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.subplots_adjust(top=0.90, hspace=0.55, wspace=0.4)
    plt.savefig(os.path.join(OUTPUT_DIR, f'Fig_PDP_1D_{tag}.png'),
                dpi=300, bbox_inches='tight')
    plt.close()
    print(f'  {tgt_name} 1D PDP 完成')


In [ ]:
print('\n' + '=' * 60)
print('9. 2D PDP 等高线图')
print('=' * 60)

FEATURE_PAIRS_2D = [
    ('Illumination intensity (μmol photons m-2 s-1)', 'Light time (h)'),
    ('temperature(℃)', 'PH'),
]

for tgt_name, y_arr in TARGETS.items():
    if tgt_name not in best_per_target:
        continue
    best = best_per_target[tgt_name]
    pipe = joblib.load(best['pipeline_path'])
    sel  = pipe.named_steps['selector']
    sel_feats = [FEATURE_COLS[i] for i in sel.get_support(indices=True)]
    tag  = tgt_name.replace('⁻', '-').replace(' ', '_').replace('/', '_')

    valid_pairs = [(f1, f2) for f1, f2 in FEATURE_PAIRS_2D
                   if f1 in sel_feats and f2 in sel_feats
                   and f1 in FEATURE_COLS and f2 in FEATURE_COLS]
    if not valid_pairs:
        print(f'  {tgt_name}: 无有效特征对，跳过')
        continue

    for f1, f2 in valid_pairs:
        idx1 = FEATURE_COLS.index(f1)
        idx2 = FEATURE_COLS.index(f2)
        fig, ax = plt.subplots(figsize=(9, 7))
        try:
            PartialDependenceDisplay.from_estimator(
                pipe, X_df, [(idx1, idx2)], kind='average', ax=ax,
                grid_resolution=40,
                contour_kw={'cmap': 'RdYlBu_r', 'alpha': 0.9,
                            'linewidths': 0.8},
            )
            ax.set_xlabel(f1.split('(')[0].strip(), fontweight='bold', fontsize=11)
            ax.set_ylabel(f2.split('(')[0].strip(), fontweight='bold', fontsize=11)
            ax.set_title(f'{tgt_name}\n2D PDP: {f1.split("(")[0].strip()} vs '
                         f'{f2.split("(")[0].strip()}', fontsize=12, fontweight='bold')
            plt.tight_layout()
            fname = (f'Fig_PDP_2D_{tag}_{f1[:12]}_vs_{f2[:12]}'
                     .replace('/', '_').replace(' ', '_'))
            plt.savefig(os.path.join(OUTPUT_DIR, f'{fname}.png'),
                        dpi=350, bbox_inches='tight')
            plt.close()
        except Exception as e:
            print(f'    2D PDP {f1} vs {f2} 失败: {e}')
            plt.close()

print('2D PDP 完成')


In [ ]:
print('\n' + '=' * 60)
print('10. 单目标培养条件优化')
print('=' * 60)

day_col   = 'time(d)'
light_col = 'Light time (h)'
dark_col  = 'Dark time (h)'
nitrogen_amount_col = 'Nitrogen amount (g L-1)'
phosphorus_amount_col = 'Phosphorus amount (mg L-1)'

# Dark time is constrained by the 24-hour photoperiod: dark = 24 - light.
# Nitrogen sources are mutually exclusive; phosphorus sources are also a
# mutually exclusive K2HPO4/KH2PO4 source choice in cultivation optimization.
base_cont_feats = [
    f for f in FEATURE_COLS
    if f not in (day_col, dark_col)
    and f not in NITROGEN_COLS
    and f not in PHOSPHORUS_COLS
]
cont_feats = base_cont_feats + [nitrogen_amount_col, phosphorus_amount_col]

base_bounds = []
for f in base_cont_feats:
    lo, hi = float(X_df[f].min()), float(X_df[f].max())
    if f == light_col:
        lo = max(0.0, lo)
        hi = min(24.0, hi)
    base_bounds.append((lo, hi))

def positive_bounds(col):
    positive_vals = X_df.loc[X_df[col] > 1e-12, col]
    lo = float(positive_vals.min()) if len(positive_vals) else 0.0
    hi = float(X_df[col].max())
    return lo, hi

nitrogen_bounds = {n_col: positive_bounds(n_col) for n_col in NITROGEN_COLS}
phosphorus_bounds = {p_col: positive_bounds(p_col) for p_col in PHOSPHORUS_COLS}

def bounds_for_sources(nitrogen_col, phosphorus_col):
    return base_bounds + [nitrogen_bounds[nitrogen_col], phosphorus_bounds[phosphorus_col]]

def condition_from_values(vals, nitrogen_col, phosphorus_col, day=OPTIMIZE_DAY):
    vals_by_feature = dict(zip(cont_feats, vals))
    light = float(vals_by_feature.get(light_col, X_df[light_col].median()))
    nitrogen_amount = float(vals_by_feature[nitrogen_amount_col])
    phosphorus_amount = float(vals_by_feature[phosphorus_amount_col])
    row = []
    for f in FEATURE_COLS:
        if f == day_col:
            row.append(day)
        elif f == dark_col:
            row.append(24.0 - light)
        elif f in NITROGEN_COLS:
            row.append(nitrogen_amount if f == nitrogen_col else 0.0)
        elif f in PHOSPHORUS_COLS:
            row.append(phosphorus_amount if f == phosphorus_col else 0.0)
        else:
            row.append(vals_by_feature[f])
    return row

def condition_dict_from_values(vals, nitrogen_col, phosphorus_col, day=OPTIMIZE_DAY):
    row = condition_from_values(vals, nitrogen_col, phosphorus_col, day)
    out = {f: float(v) for f, v in zip(FEATURE_COLS, row)}
    out['Nitrogen source'] = nitrogen_col
    out['Phosphorus source'] = phosphorus_col
    return out

optimization_results = {}

for tgt_name, y_arr in TARGETS.items():
    if tgt_name not in best_per_target:
        continue
    print(f'\n--- Target: {tgt_name} ---')
    pipe = joblib.load(best_per_target[tgt_name]['pipeline_path'])

    best_candidate = None
    for nitrogen_col in NITROGEN_COLS:
        for phosphorus_col in PHOSPHORUS_COLS:
            def objective(vals, pipe_ref=pipe, n_col=nitrogen_col, p_col=phosphorus_col):
                row = condition_from_values(vals, n_col, p_col, OPTIMIZE_DAY)
                pred = pipe_ref.predict(pd.DataFrame([row], columns=FEATURE_COLS))[0]
                return -pred

            result = differential_evolution(
                objective, bounds_for_sources(nitrogen_col, phosphorus_col),
                seed=RANDOM_SEED, maxiter=200, popsize=15, tol=1e-8,
                mutation=(0.5, 1), recombination=0.9
            )
            score = -result.fun
            if best_candidate is None or score > best_candidate['score']:
                best_candidate = {
                    'score': float(score),
                    'result': result,
                    'nitrogen_col': nitrogen_col,
                    'phosphorus_col': phosphorus_col,
                }

    result = best_candidate['result']
    best_score = best_candidate['score']
    best_nitrogen_col = best_candidate['nitrogen_col']
    best_phosphorus_col = best_candidate['phosphorus_col']
    opt_dict = condition_dict_from_values(
        result.x, best_nitrogen_col, best_phosphorus_col, OPTIMIZE_DAY
    )

    opt_row = pd.DataFrame([[opt_dict[f] for f in FEATURE_COLS]], columns=FEATURE_COLS)
    component_predictions = {}
    for comp_tgt in [tn for tn in TARGETS if 'normalized' not in tn]:
        if comp_tgt in best_per_target:
            comp_pipe = joblib.load(best_per_target[comp_tgt]['pipeline_path'])
            component_predictions[comp_tgt] = float(comp_pipe.predict(opt_row)[0])

    optimization_results[tgt_name] = {
        'predicted_value':    float(best_score),
        'component_predictions': component_predictions,
        'optimal_conditions': opt_dict,
        'nitrogen_source': best_nitrogen_col,
        'phosphorus_source': best_phosphorus_col,
        'convergence':        bool(result.success),
        'nit':                int(result.nit),
    }
    print(f'  Best predicted value: {best_score:.4f}  (converged: {result.success})')
    print(f'  Nitrogen source: {best_nitrogen_col}')
    print(f'  Phosphorus source: {best_phosphorus_col}')
    if component_predictions:
        print('  Component predictions at this optimum:')
        for comp_name, comp_val in component_predictions.items():
            print(f'    {comp_name}: {comp_val:.4f}')
    for k, v in opt_dict.items():
        if isinstance(v, (int, float, np.floating)):
            print(f'    {k}: {v:.4f}')
        else:
            print(f'    {k}: {v}')

with open(os.path.join(OUTPUT_DIR, 'single_objective_optimization.json'),
          'w', encoding='utf-8') as f:
    json.dump(optimization_results, f, indent=2, ensure_ascii=False)
print('\n单目标优化结果已保存')


In [ ]:
print('\n' + '=' * 60)
print('11. 多目标帕累托优化')
print('=' * 60)

# 加载三个最佳模型
pareto_solutions = []
pipelines_3 = {}
for tn in TARGETS:
    if tn in best_per_target:
        pipelines_3[tn] = joblib.load(best_per_target[tn]['pipeline_path'])

pareto_pipelines = {tn: pipe for tn, pipe in pipelines_3.items()
                    if 'normalized' not in tn}
if len(pareto_pipelines) < 2:
    print('Warning: fewer than 2 independent target models; skipping Pareto optimization')
else:
    # Biomass+Chl is a composite utility, so exclude it from independent Pareto targets.
    pipelines_3 = pareto_pipelines
    tgt_list = list(pipelines_3.keys())

    # Step 1: sample candidate points
    n_samples = 3000
    rng       = np.random.RandomState(RANDOM_SEED)
    cand_cont = np.zeros((n_samples, len(cont_feats)))
    cand_nitrogen = rng.choice(NITROGEN_COLS, size=n_samples)
    cand_phosphorus = rng.choice(PHOSPHORUS_COLS, size=n_samples)
    for ci, f in enumerate(cont_feats):
        if f == nitrogen_amount_col:
            for n_col in NITROGEN_COLS:
                mask = cand_nitrogen == n_col
                lo, hi = nitrogen_bounds[n_col]
                cand_cont[mask, ci] = rng.uniform(lo, hi, mask.sum())
        elif f == phosphorus_amount_col:
            for p_col in PHOSPHORUS_COLS:
                mask = cand_phosphorus == p_col
                lo, hi = phosphorus_bounds[p_col]
                cand_cont[mask, ci] = rng.uniform(lo, hi, mask.sum())
        else:
            lo, hi = base_bounds[base_cont_feats.index(f)]
            cand_cont[:, ci] = rng.uniform(lo, hi, n_samples)

    # Build the full feature matrix while enforcing dark time = 24 - light time,
    # one active nitrogen source, and one active phosphorus source per candidate.
    def build_full(cont_vals_2d, nitrogen_cols_1d=None, phosphorus_cols_1d=None,
                   day_val=OPTIMIZE_DAY):
        n  = len(cont_vals_2d)
        if nitrogen_cols_1d is None:
            nitrogen_cols_1d = np.array([NITROGEN_COLS[0]] * n)
        if phosphorus_cols_1d is None:
            phosphorus_cols_1d = np.array([PHOSPHORUS_COLS[0]] * n)
        full = np.zeros((n, len(FEATURE_COLS)))
        cont_index = {f: i for i, f in enumerate(cont_feats)}
        light_vals = cont_vals_2d[:, cont_index[light_col]]
        nitrogen_amounts = cont_vals_2d[:, cont_index[nitrogen_amount_col]]
        phosphorus_amounts = cont_vals_2d[:, cont_index[phosphorus_amount_col]]
        for ci_f, f in enumerate(FEATURE_COLS):
            if f == day_col:
                full[:, ci_f] = day_val
            elif f == dark_col:
                full[:, ci_f] = 24.0 - light_vals
            elif f in NITROGEN_COLS:
                full[:, ci_f] = np.where(nitrogen_cols_1d == f, nitrogen_amounts, 0.0)
            elif f in PHOSPHORUS_COLS:
                full[:, ci_f] = np.where(phosphorus_cols_1d == f, phosphorus_amounts, 0.0)
            else:
                full[:, ci_f] = cont_vals_2d[:, cont_index[f]]
        return pd.DataFrame(full, columns=FEATURE_COLS)

    cand_df   = build_full(cand_cont, cand_nitrogen, cand_phosphorus)
    pred_vals = {}
    for tn, pipe in pipelines_3.items():
        pred_vals[tn] = pipe.predict(cand_df)

    # 归一化各目标到[0,1]便于帕累托比较
    def norm01(arr):
        lo, hi = arr.min(), arr.max()
        return (arr - lo) / (hi - lo + 1e-12)

    norm_preds = {tn: norm01(pred_vals[tn]) for tn in tgt_list}
    pts = np.column_stack([norm_preds[tn] for tn in tgt_list])  # (n_samples, n_targets)

    # ── 步骤2: 帕累托支配筛选 ─────────────────────────────────
    n_pts     = len(pts)
    is_pareto = np.ones(n_pts, dtype=bool)
    for i in range(n_pts):
        if not is_pareto[i]:
            continue
        # 找所有支配i的点
        dominated_by = (np.all(pts >= pts[i], axis=1) &
                        np.any(pts > pts[i], axis=1))
        dominated_by[i] = False
        if dominated_by.any():
            is_pareto[i] = False
        else:
            # i支配其他点
            dominates = (np.all(pts[i] >= pts, axis=1) &
                         np.any(pts[i] > pts, axis=1))
            dominates[i] = False
            is_pareto[dominates] = False

    pareto_idx = np.where(is_pareto)[0]
    print(f'帕累托前沿点数: {len(pareto_idx)} / {n_pts}')

    # ── 步骤3: 加权优化扫描12个典型策略 ─────────────────────
    # 三目标: Biomass, Total Chl, Biomass+Chl
    if len(tgt_list) == 2:
        weights_scan = [
            (1.0, 0.0), (0.0, 1.0),
            (0.5, 0.5), (0.6, 0.4), (0.4, 0.6),
            (0.7, 0.3), (0.3, 0.7), (0.8, 0.2),
            (0.2, 0.8), (0.9, 0.1), (0.1, 0.9),
            (0.55, 0.45),
        ]
    else:
        weights_scan = [
            (1.0, 0.0, 0.0), (0.0, 1.0, 0.0), (0.0, 0.0, 1.0),
            (0.5, 0.5, 0.0), (0.5, 0.0, 0.5), (0.0, 0.5, 0.5),
            (1/3, 1/3, 1/3), (0.6, 0.2, 0.2), (0.2, 0.6, 0.2),
            (0.2, 0.2, 0.6), (0.5, 0.3, 0.2), (0.3, 0.5, 0.2),
        ]

    max_vals = {tn: pred_vals[tn].max() + 1e-12 for tn in tgt_list}

    pareto_solutions = []
    for w_tuple in weights_scan:
        w = dict(zip(tgt_list, w_tuple[:len(tgt_list)]))

        def weighted_obj(cont_vals, nitrogen_col, phosphorus_col):
            df_in = build_full(
                cont_vals.reshape(1, -1),
                np.array([nitrogen_col]),
                np.array([phosphorus_col])
            )
            total = 0.0
            for tn, pipe in pipelines_3.items():
                total -= w.get(tn, 0) * pipe.predict(df_in)[0] / max_vals[tn]
            return total

        best_weighted = None
        for nitrogen_col in NITROGEN_COLS:
            for phosphorus_col in PHOSPHORUS_COLS:
                res = differential_evolution(
                    lambda v, n_col=nitrogen_col, p_col=phosphorus_col:
                        weighted_obj(v, n_col, p_col),
                    bounds_for_sources(nitrogen_col, phosphorus_col),
                    seed=RANDOM_SEED, maxiter=100, popsize=10, tol=1e-7
                )
                score = -res.fun
                if best_weighted is None or score > best_weighted['score']:
                    best_weighted = {
                        'score': score,
                        'result': res,
                        'nitrogen_col': nitrogen_col,
                        'phosphorus_col': phosphorus_col,
                    }

        res = best_weighted['result']
        nitrogen_col = best_weighted['nitrogen_col']
        phosphorus_col = best_weighted['phosphorus_col']
        cond = condition_dict_from_values(res.x, nitrogen_col, phosphorus_col, OPTIMIZE_DAY)
        df_in = build_full(
            res.x.reshape(1, -1),
            np.array([nitrogen_col]),
            np.array([phosphorus_col])
        )
        preds = {tn: float(pipe.predict(df_in)[0]) for tn, pipe in pipelines_3.items()}
        pareto_solutions.append({
            'weights':      dict(zip(tgt_list, w_tuple[:len(tgt_list)])),
            'conditions':   cond,
            'predictions':  preds,
        })

    # ── 步骤4: 保存帕累托结果 ──────────────────────────────
    pareto_df_rows = []
    for sol in pareto_solutions:
        row = {f'w_{i}': v for i, v in enumerate(sol['weights'].values())}
        row.update({f'pred_{k.split("(")[0].strip()}': v
                    for k, v in sol['predictions'].items()})
        row.update({f'cond_{k}': v for k, v in sol['conditions'].items()})
        pareto_df_rows.append(row)
    pareto_df = pd.DataFrame(pareto_df_rows)
    pareto_df.to_csv(os.path.join(OUTPUT_DIR, 'pareto_solutions.csv'),
                     index=False, encoding='utf-8-sig')

    # ── 步骤5: 帕累托可视化 ────────────────────────────────

    # (a) Biomass vs Total Chl
    if len(tgt_list) >= 2:
        t0, t1 = tgt_list[0], tgt_list[1]
        b0, b1 = pred_vals[t0], pred_vals[t1]
        p_b0   = b0[pareto_idx]
        p_b1   = b1[pareto_idx]
        sort_i = np.argsort(p_b0)

        fig, ax = plt.subplots(figsize=(9, 7))
        ax.scatter(b0, b1, alpha=0.15, s=12, c='#BDC3C7', label='Candidate samples',
                   edgecolors='white', linewidth=0.2)
        ax.scatter(p_b0, p_b1, c='#E74C3C', s=50, zorder=5, edgecolors='darkred',
                   linewidth=0.5, label=f'Pareto front (n={len(pareto_idx)})')
        ax.plot(p_b0[sort_i], p_b1[sort_i], '--', color='#E74C3C', lw=1.5, alpha=0.8)
        ax.fill_between(p_b0[sort_i], p_b1[sort_i], alpha=0.05, color='#E74C3C')
        ax.set_xlabel(t0, fontweight='bold', fontsize=12)
        ax.set_ylabel(t1, fontweight='bold', fontsize=12)
        ax.set_title(f'Pareto Front: {t0.split("(")[0].strip()} vs '
                     f'{t1.split("(")[0].strip()}', fontsize=14, fontweight='bold')
        ax.legend(loc='upper left', fontsize=10,
                  framealpha=0.9, edgecolor='#BDBDBD')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3, linestyle='--')

        # 标注12个加权策略点
        markers = ['*', 'D', 's', '^', 'v', 'p', 'h', 'X', 'P', 'o', '<', '>']
        strategy_colors = sns.color_palette('tab10', 12)
        for si, sol in enumerate(pareto_solutions):
            px = sol['predictions'].get(t0, None)
            py = sol['predictions'].get(t1, None)
            if px is not None and py is not None and np.isfinite(px) and np.isfinite(py):
                ax.scatter(px, py, marker=markers[si % len(markers)],
                           s=140, c=[strategy_colors[si]], zorder=6,
                           edgecolors='black', linewidth=0.5)
                ax.annotate(f'S{si+1}', (px, py),
                            textcoords='offset points', xytext=(6, 6),
                            fontsize=8, fontweight='bold', color='#2C3E50',
                            bbox=dict(boxstyle='round,pad=0.2',
                                      facecolor='white', edgecolor='#BDBDBD',
                                      alpha=0.8))

        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'Fig_Pareto_biomass_chl.png'),
                    dpi=350, bbox_inches='tight')
        plt.close()

    # (b) Biomass vs Biomass+Chl (组合目标)
    if len(tgt_list) >= 3:
        t0, t2 = tgt_list[0], tgt_list[2]
        b0, b2 = pred_vals[t0], pred_vals[t2]
        p_b0   = b0[pareto_idx]
        p_b2   = b2[pareto_idx]
        sort_i = np.argsort(p_b0)

        fig, ax = plt.subplots(figsize=(9, 7))
        ax.scatter(b0, b2, alpha=0.15, s=12, c='#BDC3C7', label='Candidate samples',
                   edgecolors='white', linewidth=0.2)
        ax.scatter(p_b0, p_b2, c='#F39C12', s=50, zorder=5, edgecolors='#D68910',
                   linewidth=0.5, label=f'Pareto front (n={len(pareto_idx)})')
        ax.plot(p_b0[sort_i], p_b2[sort_i], '--', color='#F39C12', lw=1.5, alpha=0.8)
        ax.fill_between(p_b0[sort_i], p_b2[sort_i], alpha=0.05, color='#F39C12')
        ax.set_xlabel(t0, fontweight='bold', fontsize=12)
        ax.set_ylabel(t2, fontweight='bold', fontsize=12)
        ax.set_title(f'Pareto Front: {t0.split("(")[0].strip()} vs '
                     f'{t2.split("(")[0].strip()}', fontsize=14, fontweight='bold')
        ax.legend(loc='upper left', fontsize=10,
                  framealpha=0.9, edgecolor='#BDBDBD')
        ax.spines['top'].set_visible(False)
        ax.spines['right'].set_visible(False)
        ax.grid(True, alpha=0.3, linestyle='--')

        markers = ['*', 'D', 's', '^', 'v', 'p', 'h', 'X', 'P', 'o', '<', '>']
        strategy_colors = sns.color_palette('tab10', 12)
        for si, sol in enumerate(pareto_solutions):
            px = sol['predictions'].get(t0, None)
            py = sol['predictions'].get(t2, None)
            if px is not None and py is not None and np.isfinite(px) and np.isfinite(py):
                ax.scatter(px, py, marker=markers[si % len(markers)],
                           s=140, c=[strategy_colors[si]], zorder=6,
                           edgecolors='black', linewidth=0.5)
                ax.annotate(f'S{si+1}', (px, py),
                            textcoords='offset points', xytext=(6, 6),
                            fontsize=8, fontweight='bold', color='#2C3E50',
                            bbox=dict(boxstyle='round,pad=0.2',
                                      facecolor='white', edgecolor='#BDBDBD',
                                      alpha=0.8))

        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'Fig_Pareto_biomass_combined.png'),
                    dpi=350, bbox_inches='tight')
        plt.close()

    # (c) 3D 帕累托（若有三目标）
    if len(tgt_list) == 3:
        from mpl_toolkits.mplot3d import Axes3D
        b0 = pred_vals[tgt_list[0]]
        b1 = pred_vals[tgt_list[1]]
        b2 = pred_vals[tgt_list[2]]
        fig = plt.figure(figsize=(11, 9))
        ax  = fig.add_subplot(111, projection='3d')
        ax.scatter(b0, b1, b2, alpha=0.1, s=6, c='#BDC3C7', label='Candidates',
                   edgecolors='none')
        ax.scatter(b0[pareto_idx], b1[pareto_idx], b2[pareto_idx],
                   c='#E74C3C', s=55, zorder=5, edgecolors='darkred',
                   linewidth=0.5, label=f'Pareto (n={len(pareto_idx)})')
        ax.set_xlabel(tgt_list[0].split('(')[0].strip(), fontsize=10, labelpad=10)
        ax.set_ylabel(tgt_list[1].split('(')[0].strip(), fontsize=10, labelpad=10)
        ax.set_zlabel(tgt_list[2].split('(')[0].strip(), fontsize=10, labelpad=10)
        ax.set_title(f'3D Pareto Front (Day = {OPTIMIZE_DAY})',
                     fontsize=14, fontweight='bold', pad=15)
        ax.legend(fontsize=9, loc='upper left')
        ax.view_init(elev=25, azim=45)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR, 'Fig_Pareto_3D.png'),
                    dpi=350, bbox_inches='tight')
        plt.close()

    print('帕累托可视化完成')


In [ ]:
print('\n' + '=' * 60)
print('12. 模型综合对比图')
print('=' * 60)

model_names_plot = list(MODEL_CONFIG.keys())
if 'Ensemble' in summary_df['Model'].values and 'Ensemble' not in model_names_plot:
    model_names_plot.append('Ensemble')
tgt_labels       = ['Biomass\n(mg/L)', 'Total Chl\n(mg/L)', 'Biomass+Chl\n(normalized)']
metrics_plot     = [('Test_R2', 'Test R²'), ('Test_RMSE', 'Test RMSE'),
                    ('Test_MAE', 'Test MAE')]
colors_bar       = ['#3498DB', '#2ECC71', '#E74C3C', '#9B59B6']

for metric_col, metric_label in metrics_plot:
    fig, ax = plt.subplots(figsize=(11, 5.5))
    x       = np.arange(len(TARGETS))
    n_models = len(model_names_plot)
    width    = 0.8 / n_models
    for ji, mn in enumerate(model_names_plot):
        vals = []
        for tn in TARGETS:
            row = summary_df[(summary_df['Target'] == tn) & (summary_df['Model'] == mn)]
            vals.append(float(row[metric_col].values[0]) if len(row) > 0 else 0)
        offset = (ji - n_models / 2 + 0.5) * width
        bars = ax.bar(x + offset, vals, width, label=mn,
                      color=colors_bar[ji % len(colors_bar)],
                      edgecolor='white', linewidth=0.8,
                      alpha=0.9, zorder=3)
        for bar, v in zip(bars, vals):
            y_pos = bar.get_height()
            if metric_col == 'Test_R2':
                txt = f'{v:.3f}'
            else:
                txt = f'{v:.2f}'
            ax.text(bar.get_x() + bar.get_width() / 2,
                    y_pos + (ax.get_ylim()[1] - ax.get_ylim()[0]) * 0.01,
                    txt, ha='center', va='bottom', fontsize=7.5,
                    fontweight='bold')

    ax.set_xticks(x)
    ax.set_xticklabels(tgt_labels[:len(TARGETS)], fontsize=11)
    ax.set_ylabel(metric_label, fontweight='bold', fontsize=11)
    ax.set_title(f'Model Comparison — {metric_label}',
                 fontweight='bold', fontsize=14)
    ax.legend(fontsize=9, framealpha=0.9, edgecolor='#BDBDBD',
              loc='best')
    ax.spines['top'].set_visible(False)
    ax.spines['right'].set_visible(False)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_DIR, f'Fig_model_compare_{metric_col}.png'),
                dpi=350, bbox_inches='tight')
    plt.close()

print('综合对比图完成')


In [ ]:
print('\n' + '=' * 60)
print('13. ML 框架流程图')
print('=' * 60)

fig, ax = plt.subplots(figsize=(20, 7))
ax.set_xlim(0, 20); ax.set_ylim(0, 7); ax.axis('off')

box_blue   = dict(boxstyle='round,pad=0.5', facecolor='#D6EAF8', edgecolor='#2471A3', lw=2.0)
box_orange = dict(boxstyle='round,pad=0.5', facecolor='#FDEBD0', edgecolor='#CA6F1E', lw=2.0)
box_green  = dict(boxstyle='round,pad=0.5', facecolor='#D5F5E3', edgecolor='#1E8449', lw=2.0)
box_purple = dict(boxstyle='round,pad=0.5', facecolor='#E8DAEF', edgecolor='#7D3C98', lw=2.0)
box_red    = dict(boxstyle='round,pad=0.5', facecolor='#FADBD8', edgecolor='#C0392B', lw=2.0)
arrow_kw   = dict(arrowstyle='->', color='#2C3E50', lw=2.2,
                  connectionstyle='arc3,rad=0')

steps = [
    (2,  3.5, 'Data Collection\n& Preprocessing\n(samples, non-time factors)', box_blue),
    (6,  3.5, 'Model Training\nXGBoost / LightGBM / CatBoost\n/ Ensemble', box_orange),
    (10, 3.5, 'Model Interpretation\nSHAP + PDP\n(1D & 2D)', box_green),
    (14, 3.5, 'Multi-objective\nPareto Optimization\n(3 targets)', box_purple),
    (18, 3.5, 'Optimal\nCultivation\nStrategy', box_red),
]
for x, y, txt, sty in steps:
    ax.text(x, y, txt, ha='center', va='center', fontsize=10.5,
            fontweight='bold', bbox=sty)

for i in range(len(steps) - 1):
    x1 = steps[i][0] + 1.65; x2 = steps[i+1][0] - 1.65
    ax.annotate('', xy=(x2, 3.5), xytext=(x1, 3.5), arrowprops=arrow_kw)

# 子注释
sub = [
    (2,  1.5, 'RFECV feature selection\nPearson correlation analysis'),
    (6,  1.5, '5-fold cross-validation\nBayesian hyperparameter tuning\nEnsemble voting'),
    (10, 1.5, 'Feature ranking\nOptimal range identification'),
    (14, 1.5, 'Pareto front (3000 samples)\n12 weighted strategies'),
]
for x, y, txt in sub:
    ax.text(x, y, txt, ha='center', va='center', fontsize=8.5,
            style='italic', color='#566573',
            bbox=dict(boxstyle='round,pad=0.35',
                      facecolor='#F8F9F9', edgecolor='#AEB6BF', lw=1.0))

ax.set_title('Machine Learning Framework for Multi-target Optimization of '
             'Chlorella sp. Cultivation',
             fontsize=15, fontweight='bold', pad=20)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'Fig_ML_framework.png'),
            dpi=350, bbox_inches='tight')
plt.close()
print('流程图已保存')


In [ ]:
print('\n' + '=' * 60)
print('14. 表格生成')
print('=' * 60)

# Table 1: 模型性能汇总
perf_df = summary_df.copy()
perf_df.columns = ['Target', 'Model', 'Train R2', 'Val R2', 'Test R2',
                   'Train RMSE', 'Val RMSE', 'Test RMSE',
                   'Train MAE', 'Val MAE', 'Test MAE']
perf_df.to_csv(os.path.join(OUTPUT_DIR, 'Table1_model_performance.csv'),
               index=False, encoding='utf-8-sig')
print('Table 1 模型性能表已保存')

# Table 2: 单目标最优培养条件
cond_rows = []
for tgt, info in optimization_results.items():
    row = {'Target': tgt, 'Predicted value': round(info['predicted_value'], 4),
           'Converged': info['convergence']}
    for comp_tgt, comp_val in info.get('component_predictions', {}).items():
        row[f'Predicted {comp_tgt}'] = round(comp_val, 4)
    row.update({
        k: round(v, 4) if isinstance(v, (int, float, np.integer, np.floating)) else v
        for k, v in info['optimal_conditions'].items()
    })
    cond_rows.append(row)
pd.DataFrame(cond_rows).to_csv(
    os.path.join(OUTPUT_DIR, 'Table2_optimal_conditions.csv'),
    index=False, encoding='utf-8-sig')
print('Table 2 最优培养条件表已保存')

# Table 3: 帕累托12策略摘要
if 'pareto_solutions' in dir() and pareto_solutions:
    par_rows = []
    for si, sol in enumerate(pareto_solutions):
        row = {f'Strategy S{si+1}': ''}
        row['Weights'] = ' / '.join(
            [f'{v:.2f}' for v in sol['weights'].values()])
        for tn, pv in sol['predictions'].items():
            row[tn] = round(pv, 4)
        par_rows.append(row)
    pd.DataFrame(par_rows).to_csv(
        os.path.join(OUTPUT_DIR, 'Table3_pareto_strategies.csv'),
        index=False, encoding='utf-8-sig')
    print('Table 3 帕累托策略表已保存')



In [ ]:
import os, shutil

SRC_DIR   = OUTPUT_DIR
DESKTOP   = os.path.join(os.path.expanduser('~'), 'Desktop')
DEST_ROOT = os.path.join(DESKTOP, 'chlorella_results_分类')
os.makedirs(DEST_ROOT, exist_ok=True)

CATEGORIES = [
    ('01_PCC与数据分布',     lambda n: n.startswith('Fig_PCC') or n.startswith('Fig_data_distribution')),
    ('02_模型散点与残差图',   lambda n: n.startswith('Fig_scatter') or n.startswith('Fig_residual')),
    ('03_SHAP分析',          lambda n: n.startswith('Fig_SHAP')),
    ('04_偏依赖图PDP',       lambda n: n.startswith('Fig_PDP')),
    ('05_帕累托优化图',      lambda n: n.startswith('Fig_Pareto')),
    ('06_模型对比与框架图',   lambda n: n.startswith('Fig_model_compare') or n.startswith('Fig_ML_framework')),
    ('07_其他图片',          lambda n: n.endswith(('.png','.jpg','.svg'))),
    ('08_数据表格',          lambda n: n.startswith('Table') and n.endswith('.csv')),
    ('09_帕累托与对比数据',   lambda n: n in ('pareto_solutions.csv','model_comparison_all.csv')),
    ('10_模型指标JSON',      lambda n: n.endswith('.json')),
    ('11_保存的模型',        lambda n: n.endswith(('.joblib','.pkl'))),
]

for fname in sorted(os.listdir(SRC_DIR)):
    src = os.path.join(SRC_DIR, fname)
    if not os.path.isfile(src): continue
    for folder, fn in CATEGORIES:
        if fn(fname):
            dest = os.path.join(DEST_ROOT, folder)
            os.makedirs(dest, exist_ok=True)
            shutil.copy2(src, os.path.join(dest, fname))
            print(f'  {fname} -> {folder}/')
            break
    else:
        d = os.path.join(DEST_ROOT, '99_未分类')
        os.makedirs(d, exist_ok=True)
        shutil.copy2(src, os.path.join(d, fname))

print(f'\n分类完成! 请在桌面查看: {DEST_ROOT}')
